In [4]:
# Pageview Table
import os
import re
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# load
df_pageview = pd.read_csv(r'C:\Users\dell\Desktop\smart-home-product-analysis\datasets\raw_data\pageview_table.csv')

# quick look
print(df_pageview.shape)
print(df_pageview.duplicated().sum())
print(df_pageview['event_id'].nunique())
display(df_pageview.head(5))
display(df_pageview.isnull().sum().rename('missing').to_frame())

# checking uuid format on all three id columns since all three are used for joining
uuid_pattern = r'^[0-9a-f]{8}-[0-9a-f]{4}-[0-9a-f]{4}-[0-9a-f]{4}-[0-9a-f]{12}$'
invalid_events   = df_pageview[~df_pageview['event_id'].str.match(uuid_pattern, na=False)]
invalid_users    = df_pageview[~df_pageview['user_id'].str.match(uuid_pattern, na=False)]
invalid_sessions = df_pageview[~df_pageview['session_id'].str.match(uuid_pattern, na=False)]
print(f"invalid event_ids  : {len(invalid_events)}")
print(f"invalid user_ids   : {len(invalid_users)}")
print(f"invalid session_ids: {len(invalid_sessions)}")

# drop bad ids and duplicates
df_pageview = df_pageview[df_pageview['event_id'].str.match(uuid_pattern, na=False)]
df_pageview = df_pageview[df_pageview['user_id'].str.match(uuid_pattern, na=False)]
df_pageview = df_pageview[df_pageview['session_id'].str.match(uuid_pattern, na=False)]
df_pageview = df_pageview.drop_duplicates(subset=['event_id'], keep='first')

# time came in as string, converting to datetime
df_pageview['time'] = pd.to_datetime(df_pageview['time'], format='mixed', dayfirst=True)

# strip and lowercase everything to avoid groupby mismatches later
str_cols = df_pageview.select_dtypes(include='object').columns
df_pageview[str_cols] = df_pageview[str_cols].apply(lambda x: x.str.strip().str.lower())

# pulling out page category from path
# e.g. /products/video-doorbell-pro-2 becomes products
# this makes it easy to group and filter pages in funnel analysis
df_pageview['page_category'] = df_pageview['path'].apply(
    lambda x: x.split('/')[1] if isinstance(x, str) and len(x.split('/')) > 1 else 'other'
)

# hash and query are not needed for funnel so leaving them as is
# previous_page being empty just means it was the first page of that session, nothing to fix

# final check
print(df_pageview.shape)
display(df_pageview.isnull().sum().rename('missing').to_frame())
display(df_pageview.head(10))
display(df_pageview.dtypes.rename('dtype').to_frame())

# save
CLEANED = r'C:\Users\dell\Desktop\smart-home-product-analysis\datasets\cleaned_data'
os.makedirs(CLEANED, exist_ok=True)
df_pageview.to_csv(os.path.join(CLEANED, 'pageview_cleaned.csv'), index=False)
print("saved pageview_cleaned.csv")

(354456, 9)
0
354456


,event_id,user_id,session_id,time,domain,path,hash,query,previous_page
0,6bb16020-e292-4edd-84a1-9a907aa9e971,9d415f62-5d38-436d-a443-336b2b759568,a32f32ca-14ed-44d5-afb4-070d082e88e7,2025-01-01 23:17:31,ring.com,/products/video-doorbell-pro-2,NaN,utm_source=impact&utm_medium=affiliate&utm_cam...,NaN
1,2c181503-a4f1-4068-96bf-d2e805f9beb1,a71251a4-f215-4046-85fb-51206780e6c8,f43c11f7-9553-4408-ae10-98fd285ba383,2025-01-01 22:42:04,ring.com,/products/stick-up-cam-battery,NaN,utm_source=ring.com&utm_medium=internal&utm_ca...,NaN
2,5562a32f-6682-4039-8de6-c810e06e60f6,a8e69945-e3f7-4458-8ea9-883062d75c71,14d9b0a5-d40c-43aa-9f02-c4a24f3f0c3a,2025-01-01 12:06:30,ring.com,/products/video-doorbell-pro-2,NaN,li_fat_id=TH8xIZM1JRcoreogrNwwmq6O&utm_source=...,NaN
3,2bbc4734-15a8-4e7b-a3a6-bab825d8f980,a71251a4-f215-4046-85fb-51206780e6c8,f829d83e-fae0-4ce2-bd2a-9b9b2b5567a8,2025-01-01 02:14:14,ring.com,/products/ring-alarm-8-piece,NaN,ttclid=y4CqpIqK3yn9FfcgMXAdx9G8&utm_source=tik...,NaN
4,d6af767f-959a-4794-b382-b3512156dfdd,a71251a4-f215-4046-85fb-51206780e6c8,f829d83e-fae0-4ce2-bd2a-9b9b2b5567a8,2025-01-01 02:14:24,ring.com,/cart,NaN,NaN,/products/ring-alarm-8-piece


,missing
event_id,0
user_id,0
session_id,0
time,0
domain,0
path,0
hash,354456
query,219637
previous_page,146000


invalid event_ids  : 0
invalid user_ids   : 0
invalid session_ids: 0
(354456, 10)


,missing
event_id,0
user_id,0
session_id,0
time,0
domain,0
path,0
hash,354456
query,219637
previous_page,146000
page_category,0


,event_id,user_id,session_id,time,domain,path,hash,query,previous_page,page_category
0,6bb16020-e292-4edd-84a1-9a907aa9e971,9d415f62-5d38-436d-a443-336b2b759568,a32f32ca-14ed-44d5-afb4-070d082e88e7,2025-01-01 23:17:31,ring.com,/products/video-doorbell-pro-2,NaN,utm_source=impact&utm_medium=affiliate&utm_cam...,NaN,products
1,2c181503-a4f1-4068-96bf-d2e805f9beb1,a71251a4-f215-4046-85fb-51206780e6c8,f43c11f7-9553-4408-ae10-98fd285ba383,2025-01-01 22:42:04,ring.com,/products/stick-up-cam-battery,NaN,utm_source=ring.com&utm_medium=internal&utm_ca...,NaN,products
2,5562a32f-6682-4039-8de6-c810e06e60f6,a8e69945-e3f7-4458-8ea9-883062d75c71,14d9b0a5-d40c-43aa-9f02-c4a24f3f0c3a,2025-01-01 12:06:30,ring.com,/products/video-doorbell-pro-2,NaN,li_fat_id=th8xizm1jrcoreogrnwwmq6o&utm_source=...,NaN,products
3,2bbc4734-15a8-4e7b-a3a6-bab825d8f980,a71251a4-f215-4046-85fb-51206780e6c8,f829d83e-fae0-4ce2-bd2a-9b9b2b5567a8,2025-01-01 02:14:14,ring.com,/products/ring-alarm-8-piece,NaN,ttclid=y4cqpiqk3yn9ffcgmxadx9g8&utm_source=tik...,NaN,products
4,d6af767f-959a-4794-b382-b3512156dfdd,a71251a4-f215-4046-85fb-51206780e6c8,f829d83e-fae0-4ce2-bd2a-9b9b2b5567a8,2025-01-01 02:14:24,ring.com,/cart,NaN,NaN,/products/ring-alarm-8-piece,cart
5,bc4d59e8-b408-45d8-b04f-dad7d466fc78,a71251a4-f215-4046-85fb-51206780e6c8,f829d83e-fae0-4ce2-bd2a-9b9b2b5567a8,2025-01-01 02:14:39,ring.com,/checkout/payment,NaN,NaN,/cart,checkout
6,8671e40a-ae2e-463b-be6a-2a5f0288aed8,a71251a4-f215-4046-85fb-51206780e6c8,f829d83e-fae0-4ce2-bd2a-9b9b2b5567a8,2025-01-01 02:14:48,ring.com,/category/security-cameras,NaN,NaN,/checkout/payment,category
7,733d17a0-b564-47b8-bc72-084b8d4db74a,a71251a4-f215-4046-85fb-51206780e6c8,f829d83e-fae0-4ce2-bd2a-9b9b2b5567a8,2025-01-01 02:15:00,ring.com,/cart,NaN,NaN,/category/security-cameras,cart
8,8cfcb187-1b67-4d36-8350-fc6dbffa505b,a71251a4-f215-4046-85fb-51206780e6c8,f829d83e-fae0-4ce2-bd2a-9b9b2b5567a8,2025-01-01 02:15:19,ring.com,/cart,NaN,NaN,/cart,cart
9,d7552d1b-4bd4-4932-a29a-0c62f38fea4d,a71251a4-f215-4046-85fb-51206780e6c8,f829d83e-fae0-4ce2-bd2a-9b9b2b5567a8,2025-01-01 02:15:28,ring.com,/cart,NaN,NaN,/cart,cart


,dtype
event_id,str
user_id,str
session_id,str
time,datetime64[us]
domain,str
path,str
hash,float64
query,str
previous_page,str
page_category,str


saved pageview_cleaned.csv
